# Stage 6 — social/network forensic analysis

This notebook builds SIM–contact relationship evidence from canonical CDRs. It measures network participation and counterparties; it does not infer intent from a contact alone.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/social/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])
fraud=set(pd.read_csv(PROCESSED/'fraudulent_users.csv',dtype={'src_number':'string'})['src_number'].dropna().tolist())
cdr['src_known_fraud']=cdr.src_number.isin(fraud); cdr['dst_known_fraud']=cdr.dst_number.isin(fraud)
print('canonical CDRs:',len(cdr),'fraud labels:',len(fraud))

canonical CDRs: 86873 fraud labels: 200


C:\Users\salai_wciilqo\AppData\Local\Temp\ipykernel_424\2919121718.py:8: DtypeWarning: Columns (3: src_imei, 7: dst_imei) have mixed types. Specify dtype option on import or set low_memory=False.
  cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])


In [2]:
edges=cdr[['record_id','src_number','dst_number','duration','cdr_type','src_known_fraud','dst_known_fraud']].copy()
edges['self_contact']=edges.src_number==edges.dst_number

source=edges[['src_number','dst_number','record_id','duration','dst_known_fraud','self_contact']].rename(columns={'src_number':'sim','dst_number':'contact'})
source['direction']='outgoing'
destination=edges[['dst_number','src_number','record_id','duration','src_known_fraud','self_contact']].rename(columns={'dst_number':'sim','src_number':'contact','src_known_fraud':'contact_known_fraud'})
destination['direction']='incoming'; destination['dst_known_fraud']=False
source['contact_known_fraud']=source['dst_known_fraud']
endpoints=pd.concat([source[['sim','contact','record_id','duration','direction','contact_known_fraud','self_contact']],destination[['sim','contact','record_id','duration','direction','contact_known_fraud','self_contact']]],ignore_index=True)

features=endpoints.groupby('sim').agg(unique_contacts=('contact','nunique'),outgoing_degree=('contact',lambda x: x[endpoints.loc[x.index,'direction'].eq('outgoing')].nunique()),incoming_degree=('contact',lambda x: x[endpoints.loc[x.index,'direction'].eq('incoming')].nunique()),total_relationship_events=('record_id','nunique'),self_contact_events=('self_contact','sum'),contacts_known_fraud=('contact_known_fraud','sum')).reset_index()
features['known_fraud']=features.sim.isin(fraud)
features['contact_fraud_rate']=features.contacts_known_fraud/features.total_relationship_events.replace(0,np.nan)
features['events_per_contact']=features.total_relationship_events/features.unique_contacts.replace(0,np.nan)
features.to_csv(OUTPUT/'social_network_features.csv',index=False)
features.head()

,sim,unique_contacts,outgoing_degree,incoming_degree,total_relationship_events,self_contact_events,contacts_known_fraud,known_fraud,contact_fraud_rate,events_per_contact
0,+11102241212,1,0,1,1,0,0,False,0.0,1.0
1,+11124222602,1,0,1,1,0,0,False,0.0,1.0
2,+11185236570,1,1,0,1,0,0,False,0.0,1.0
3,+11201660070,1,0,1,1,0,0,False,0.0,1.0
4,+11278389544,1,0,1,1,0,0,False,0.0,1.0


In [3]:
# Reciprocal relationships: contacts appearing in both directions for a SIM.
out_pairs=set(zip(edges.src_number,edges.dst_number)); reciprocal_counts={}
for sim in features.sim:
    contacts=set(edges.loc[edges.src_number.eq(sim),'dst_number'])
    reciprocal_counts[sim]=sum((contact,sim) in out_pairs for contact in contacts)
features['reciprocal_contacts']=features.sim.map(reciprocal_counts)
features['reciprocity_rate']=features.reciprocal_contacts/features.unique_contacts.replace(0,np.nan)
features.to_csv(OUTPUT/'social_network_features.csv',index=False)

numeric=['unique_contacts','outgoing_degree','incoming_degree','total_relationship_events','self_contact_events','contacts_known_fraud','contact_fraud_rate','events_per_contact','reciprocal_contacts','reciprocity_rate']
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')
comparison=features.groupby('known_fraud',observed=False)[numeric].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']; comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'social_group_median_comparison.csv')
print(comparison)

                          known_legitimate_median known_fraudulent_median  \
unique_contacts                               1.0                     3.0   
outgoing_degree                               1.0                     3.0   
incoming_degree                               1.0                     0.0   
total_relationship_events                     2.0                     3.0   
self_contact_events                           0.0                     0.0   
contacts_known_fraud                          0.0                     0.0   
contact_fraud_rate                            0.0                     0.0   
events_per_contact                            1.0                     1.0   
reciprocal_contacts                           0.0                     0.0   
reciprocity_rate                              0.0                     0.0   

                          difference  
unique_contacts                  2.0  
outgoing_degree                  2.0  
incoming_degree                 -1.

In [4]:
# Contact-level edge table for later graph inspection.
edge_summary=edges.groupby(['src_number','dst_number']).agg(event_count=('record_id','nunique'),total_duration=('duration','sum'),first_seen=('record_id','min'),src_known_fraud=('src_known_fraud','max'),dst_known_fraud=('dst_known_fraud','max')).reset_index()
edge_summary.to_csv(OUTPUT/'contact_edge_summary.csv',index=False)
print('unique directed edges:',len(edge_summary))

unique directed edges: 8745


Interpretation rule: high degree, repeated contacts, or reciprocity are network evidence. They do not independently establish fraud. Contact-level labels are retained for later graph analysis.